In [40]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,cross_validate,StratifiedKFold
from sklearn.metrics import make_scorer,cohen_kappa_score,matthews_corrcoef,precision_score,accuracy_score,recall_score,f1_score,roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE,ADASYN
from imblearn.pipeline import Pipeline
from scipy.stats import wilcoxon

In [41]:
data=pd.read_csv("bank-additional-full.csv",sep=';')

In [42]:
data=data.drop("duration",axis=1)

In [43]:
allColumns=data.select_dtypes(include='object').columns
for col in allColumns:
  data[col]=data[col].replace('unknown',data[col].mode()[0])

In [44]:
data['y']=data['y'].map({'yes':1,'no':0})

In [45]:
data=pd.get_dummies(data,drop_first=True)

In [46]:
X=data.drop('y',axis=1)
y=data['y']

In [47]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [48]:
scalar=StandardScaler()
scalar.fit_transform(X_train)
scalar.transform(X_test)

array([[-0.77033007,  0.88631588,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [-0.28972159, -0.56702251,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [ 3.17065947, -0.20368791,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       ...,
       [-0.67420837, -0.56702251,  0.19658384, ..., -0.4964409 ,
        -2.50346033, -0.18627755],
       [ 0.38313029,  1.61298507,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [ 0.19088689,  0.88631588,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755]])

In [49]:
kappa=make_scorer(cohen_kappa_score)
mcc=make_scorer(matthews_corrcoef)
scoring={
    'accuracy':'accuracy',
    'precision':'precision',
    'recall':'recall',
    'f1':'f1',
    'roc_auc':'roc_auc',
    'kappa':kappa,
    'mcc':mcc
}

In [50]:
smote=SMOTE(random_state=42)
adasyn=ADASYN(random_state=42)

In [51]:
Random_Forest=RandomForestClassifier(n_estimators=50,max_depth=10,random_state=42)
Decision_Tree=DecisionTreeClassifier(min_samples_split=2,max_depth=5,random_state=42)

In [52]:
rf_pipeline=Pipeline([
    ('sampler',smote),
    ('model',Random_Forest)
])

dt_pipeline=Pipeline([
    ('sampler',adasyn),
    ('model',Decision_Tree)
])

In [53]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [54]:
rf_results=cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    n_jobs=-1
)

dt_results=cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    n_jobs=-1
)

In [55]:
print("========== RANDOM FOREST SCORES ==========")

for metric in scoring.keys():

    print(f"\n{metric.upper()} Scores:")

    print(rf_results[f'test_{metric}'])

    print(f"Mean {metric}: {np.mean(rf_results[f'test_{metric}']):.4f}")

========== RANDOM FOREST SCORES ==========

ACCURACY Scores:
[0.86267071 0.86084977 0.86282246 0.8629742  0.86494689]
Mean accuracy: 0.8629

PRECISION Scores:
[0.41890547 0.41513094 0.41834677 0.42094862 0.42538071]
Mean precision: 0.4197

RECALL Scores:
[0.56738544 0.57681941 0.55929919 0.57335128 0.56393001]
Mean recall: 0.5682

F1 Scores:
[0.48196909 0.48279752 0.47866205 0.48547009 0.4849537 ]
Mean f1: 0.4828

ROC_AUC Scores:
[0.77984226 0.77876003 0.78249792 0.78326567 0.78621711]
Mean roc_auc: 0.7821

KAPPA Scores:
[0.40487322 0.40486599 0.40156698 0.40856729 0.40898707]
Mean kappa: 0.4058

MCC Scores:
[0.41106764 0.41218101 0.40718182 0.41506007 0.41437271]
Mean mcc: 0.4120


In [56]:
print("========== DECISION TREE SCORES ==========")

for metric in scoring.keys():

    print(f"\n{metric.upper()} Scores:")

    print(dt_results[f'test_{metric}'])

    print(f"Mean {metric}: {np.mean(rf_results[f'test_{metric}']):.4f}")

========== DECISION TREE SCORES ==========

ACCURACY Scores:
[0.77693475 0.85326252 0.86631259 0.7737481  0.80257967]
Mean accuracy: 0.8629

PRECISION Scores:
[0.28072289 0.3945642  0.42502697 0.28179697 0.30407303]
Mean precision: 0.4197

RECALL Scores:
[0.62803235 0.56738544 0.5309973  0.65006729 0.58277254]
Mean recall: 0.5682

F1 Scores:
[0.38800999 0.46545053 0.47213901 0.39316239 0.39963083]
Mean f1: 0.4828

ROC_AUC Scores:
[0.73644467 0.7332888  0.73399792 0.74584924 0.75807105]
Mean roc_auc: 0.7821

KAPPA Scores:
[0.27521409 0.38357489 0.39667831 0.27988572 0.29519389]
Mean kappa: 0.4058

MCC Scores:
[0.30863881 0.39208153 0.3999006  0.31689411 0.31759764]
Mean mcc: 0.4120


In [58]:
wilcoxon_results = []
alpha = 0.05
print("\n========== WILCOXON TEST RESULTS ==========")
for metric in scoring.keys():
    print(f"\n===== {metric.upper()} =====")
    # Wilcoxon Test
    statistic, p_value = wilcoxon(
        rf_results[f'test_{metric}'],
        dt_results[f'test_{metric}']
    )
    rf_mean = np.mean(
        rf_results[f'test_{metric}']
    )
    dt_mean = np.mean(
        dt_results[f'test_{metric}']
    )
    # Significance Result
    if p_value < alpha:

        result = "Significant"
    else:
        result = "Not Significant"
    wilcoxon_results.append({
        'Metric': metric,
        'RF_Mean': rf_mean,
        'DT_Mean': dt_mean,
        'Statistic': statistic,
        'P_Value': p_value,
        'Result': result
    })

    print(f"RF Mean Score : {rf_mean:.4f}")
    print(f"DT Mean Score : {dt_mean:.4f}")
    print(f"Statistic     : {statistic:.4f}")
    print(f"P-Value       : {p_value:.6f}")
    print(f"Result        : {result}")


wilcoxon_df = pd.DataFrame(wilcoxon_results)
print("\n========== FINAL COMPARISON TABLE ==========")

print(wilcoxon_df)

wilcoxon_df.to_csv(
    "Wilcoxon_Model_Comparison.csv",
    index=False
)



========== WILCOXON TEST RESULTS ==========

===== ACCURACY =====
RF Mean Score : 0.8629
DT Mean Score : 0.8146
Statistic     : 1.0000
P-Value       : 0.125000
Result        : Not Significant

===== PRECISION =====
RF Mean Score : 0.4197
DT Mean Score : 0.3372
Statistic     : 1.0000
P-Value       : 0.125000
Result        : Not Significant

===== RECALL =====
RF Mean Score : 0.5682
DT Mean Score : 0.5919
Statistic     : 4.0000
P-Value       : 0.437500
Result        : Not Significant

===== F1 =====
RF Mean Score : 0.4828
DT Mean Score : 0.4237
Statistic     : 0.0000
P-Value       : 0.062500
Result        : Not Significant

===== ROC_AUC =====
RF Mean Score : 0.7821
DT Mean Score : 0.7415
Statistic     : 0.0000
P-Value       : 0.062500
Result        : Not Significant

===== KAPPA =====
RF Mean Score : 0.4058
DT Mean Score : 0.3261
Statistic     : 0.0000
P-Value       : 0.062500
Result        : Not Significant

===== MCC =====
RF Mean Score : 0.4120
DT Mean Score : 0.3470
Statistic     :